### Tools

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool
from langchain_core.tools import Tool


llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.7,
    google_api_key=os.environ["GOOFLE_API_KEY"]
)

In [33]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search_duckduckgo = Tool(
    name="duckduckgo_search",
    func=search.run,
    description="Search the web using DuckDuckGo"
)


In [34]:
from langchain_community.retrievers import ArxivRetriever

retriever = ArxivRetriever(
    load_max_docs=2,
    get_full_documents=True,
)
docs = retriever.invoke("Transformer")

In [35]:
for i in range(len(docs)):
    print(docs[i])

page_content='PyramidTNT: Improved Transformer-in-Transformer Baselines
with Pyramid Architecture
Kai Han, Jianyuan Guo, Yehui Tang, Yunhe Wang
Huawei Noah’s Ark Lab
{kai.han,jianyuan.guo,tangyehui,yunhe.wang}@huawei.com
Abstract
Transformer networks have achieved great progress for
computer vision tasks. Transformer-in-Transformer (TNT)
architecture utilizes inner transformer and outer trans-
former to extract both local and global representations.
In this work, we present new TNT baselines by introduc-
ing two advanced designs: 1) pyramid architecture, and
2) convolutional stem.
The new “PyramidTNT” signiﬁ-
cantly improves the original TNT by establishing hierar-
chical representations. PyramidTNT achieves better per-
formances than the previous state-of-the-art vision trans-
formers such as Swin Transformer. We hope this new base-
line will be helpful to the further research and application
of vision transformer. Code will be available at https:
//github.com/huawei-noah/CV-Backbones

In [36]:
for i, doc in enumerate(docs):
    print(f"Document {i+1}")
    print(doc.page_content[:500])

Document 1
PyramidTNT: Improved Transformer-in-Transformer Baselines
with Pyramid Architecture
Kai Han, Jianyuan Guo, Yehui Tang, Yunhe Wang
Huawei Noah’s Ark Lab
{kai.han,jianyuan.guo,tangyehui,yunhe.wang}@huawei.com
Abstract
Transformer networks have achieved great progress for
computer vision tasks. Transformer-in-Transformer (TNT)
architecture utilizes inner transformer and outer trans-
former to extract both local and global representations.
In this work, we present new TNT baselines by introduc-
ing 
Document 2
Learning to Cluster Faces via Transformer
Jinxing Ye∗1, Xiaojiang Peng*2, Baigui Sun1, Kai Wang1,3, Xiuyu Sun1, Hao Li †1, and Hanqing Wu1
1Alibaba Group
2Shenzhen Technology University, China
3National University of Singapore, Singapore
Abstract
Face clustering is an useful tool for applications like au-
tomatic face annotation and retrieval. The main challenge
is that it is difﬁcult to cluster images from the same iden-
tity with different face poses, occlusions, and ima

### Custom Tools

In [37]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query: str):

    """This tool allows you to search Wikipedia for information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

In [38]:
@tool
def personal_info(name: str):

    """Use this tool to get personal information about Alice, Bob, or Charlie. 
    """

    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    return info.get(name, "No information available for this person.")

In [39]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool
def arxiv_tool(query: str) -> str:

    """"This tool allows you to query the arXiv database for research papers."""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)
    

In [40]:
from langchain_community.tools import DuckDuckGoSearchRun

search_duckduckgo = DuckDuckGoSearchRun(name="search_duckduckgo")
tools = [search_duckduckgo, arxiv_tool, wiki_tool, personal_info]

llm_with_tools = llm.bind_tools(tools)

In [41]:
response = llm_with_tools.invoke("What is the latest news on AI?")
response.tool_calls

[{'name': 'search_duckduckgo',
  'args': {'query': 'latest news on AI November 2024'},
  'id': 'f6475df5-c905-46b5-b93e-5b68cb63834e',
  'type': 'tool_call'}]